# Analisis EDA

### Importacion de librerias: 
Aqui se cargan las librerias necesarias para el análisis exploratorio de datos (EDA).

In [1]:
import pandas as pd
import numpy as np
import yaml
import datetime
import matplotlib.pyplot as plt
import os
import re


In [2]:


# carga de configuración
with open("../config/config.yaml", "r") as file:
    config = yaml.safe_load(file)

# cambiar el directorio de trabajo al nivel superior del proyecto
os.chdir("../")



# Descripción de los datos y transformaciones - Proyecciones_poblacion_dane(Tabla1)

In [3]:
# cargar datos
# obligar a leer como texto las columnas especificadas
columns = ['DP', 'DPNOM', 'MPIO', 'DPMP', 'AÑO', 'ÁREA GEOGRÁFICA']
df_proyecciones_poblacion_dane = pd.read_excel(config["sources"]["proyecciones_poblacion_dane"]["path"], sheet_name=config["sources"]["proyecciones_poblacion_dane"]["sheetname"], dtype={col: str for col in columns})


# df_proyecciones_poblacion_dane = pd.read_excel(config["sources"]["proyecciones_poblacion_dane"]["path"], sheet_name=config["sources"]["proyecciones_poblacion_dane"]["sheetname"])

In [4]:
df_proyecciones_poblacion_dane.head()

,DP,DPNOM,MPIO,DPMP,AÑO,ÁREA GEOGRÁFICA,Total,Hombres,Mujeres,Hombres 0 años,...,Total 91 años,Total 92 años,Total 93 años,Total 94 años,Total 95 años,Total 96 años,Total 97 años,Total 98 años,Total 99 años,Total 100 años y más
0,05,Antioquia,05001,Medellín,2018,Cabecera Municipal,2382405,1118868,1263537,13512,...,2027,1624,1282,1001,775,599,458,331,223,798
1,05,Antioquia,05001,Medellín,2018,Centros Poblados y Rural Disperso,44728,22475,22253,304,...,25,20,16,12,9,7,5,3,1,9
2,05,Antioquia,05001,Medellín,2018,Total,2427133,1141343,1285790,13816,...,2052,1644,1298,1013,784,606,463,334,224,807
3,05,Antioquia,05002,Abejorral,2018,Cabecera Municipal,7669,3646,4023,38,...,8,6,5,5,3,2,2,0,1,8
4,05,Antioquia,05002,Abejorral,2018,Centros Poblados y Rural Disperso,12699,6848,5851,96,...,6,4,3,2,1,1,2,1,0,2


In [5]:
# renombrar las columnas de acuerdo a un listado
columns_rename = {
    "DP": "COD_DPTO",
    "DPNOM": "NOM_DPTO",
    "MPIO": "COD_MPIO",
    "DPMP": "NOM_MPIO",
    "AÑO": "ANIO",
    "Total": "TOTAL_POBLACION",    # Agregar más columnas según sea necesario
    "Hombres": "TOTAL_HOMBRES",
    "Mujeres": "TOTAL_MUJERES"
}

df_proyecciones_poblacion_dane = df_proyecciones_poblacion_dane.rename(columns=columns_rename)

In [6]:
df_proyecciones_poblacion_dane.head()

,COD_DPTO,NOM_DPTO,COD_MPIO,NOM_MPIO,ANIO,ÁREA GEOGRÁFICA,TOTAL_POBLACION,TOTAL_HOMBRES,TOTAL_MUJERES,Hombres 0 años,...,Total 91 años,Total 92 años,Total 93 años,Total 94 años,Total 95 años,Total 96 años,Total 97 años,Total 98 años,Total 99 años,Total 100 años y más
0,05,Antioquia,05001,Medellín,2018,Cabecera Municipal,2382405,1118868,1263537,13512,...,2027,1624,1282,1001,775,599,458,331,223,798
1,05,Antioquia,05001,Medellín,2018,Centros Poblados y Rural Disperso,44728,22475,22253,304,...,25,20,16,12,9,7,5,3,1,9
2,05,Antioquia,05001,Medellín,2018,Total,2427133,1141343,1285790,13816,...,2052,1644,1298,1013,784,606,463,334,224,807
3,05,Antioquia,05002,Abejorral,2018,Cabecera Municipal,7669,3646,4023,38,...,8,6,5,5,3,2,2,0,1,8
4,05,Antioquia,05002,Abejorral,2018,Centros Poblados y Rural Disperso,12699,6848,5851,96,...,6,4,3,2,1,1,2,1,0,2


In [7]:
# calcular poblacion por sexo y rango de edad (cada columna trae sexo y edad en el nombre)
# \s+ porque 'Total 100 años  y más' viene con doble espacio en la fuente
patron_edad = r"^(Hombres|Mujeres|Total) (\d+) años?(?:\s+y más)?$"

def cols_edad(sexo, edad_min, edad_max):
    return [c for c in df_proyecciones_poblacion_dane.columns
            if (m := re.match(patron_edad, c)) and m.group(1) == sexo and edad_min <= int(m.group(2)) <= edad_max]

df_proyecciones_poblacion_dane["HOMBRES_65_MAS"] = df_proyecciones_poblacion_dane[cols_edad("Hombres", 65, 100)].sum(axis=1)
df_proyecciones_poblacion_dane["MUJERES_65_MAS"] = df_proyecciones_poblacion_dane[cols_edad("Mujeres", 65, 100)].sum(axis=1)
df_proyecciones_poblacion_dane["HOMBRES_0_15"] = df_proyecciones_poblacion_dane[cols_edad("Hombres", 0, 15)].sum(axis=1)
df_proyecciones_poblacion_dane["MUJERES_0_15"] = df_proyecciones_poblacion_dane[cols_edad("Mujeres", 0, 15)].sum(axis=1)

df_proyecciones_poblacion_dane["TOTAL_65_MAS"] = df_proyecciones_poblacion_dane["HOMBRES_65_MAS"] + df_proyecciones_poblacion_dane["MUJERES_65_MAS"]
df_proyecciones_poblacion_dane["TOTAL_0_15"] = df_proyecciones_poblacion_dane["HOMBRES_0_15"] + df_proyecciones_poblacion_dane["MUJERES_0_15"]

# eliminar el detalle por edad (0 a 100 y más) de hombres, mujeres y total
columnas_detalle_edad = [c for c in df_proyecciones_poblacion_dane.columns if re.match(patron_edad, c)]
df_proyecciones_poblacion_dane = df_proyecciones_poblacion_dane.drop(columns=columnas_detalle_edad)

df_proyecciones_poblacion_dane.head()

/var/folders/ym/k0p7fb7s30j873yzly7hngv00000gn/T/ipykernel_18540/1844214752.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_proyecciones_poblacion_dane["HOMBRES_65_MAS"] = df_proyecciones_poblacion_dane[cols_edad("Hombres", 65, 100)].sum(axis=1)
/var/folders/ym/k0p7fb7s30j873yzly7hngv00000gn/T/ipykernel_18540/1844214752.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_proyecciones_poblacion_dane["MUJERES_65_MAS"] = df_proyecciones_poblacion_dane[cols_edad("Mujeres", 65, 100)].sum(axis=1)
/var/folders/ym/k0p7fb7s30j

,COD_DPTO,NOM_DPTO,COD_MPIO,NOM_MPIO,ANIO,ÁREA GEOGRÁFICA,TOTAL_POBLACION,TOTAL_HOMBRES,TOTAL_MUJERES,HOMBRES_65_MAS,MUJERES_65_MAS,HOMBRES_0_15,MUJERES_0_15,TOTAL_65_MAS,TOTAL_0_15
0,05,Antioquia,05001,Medellín,2018,Cabecera Municipal,2382405,1118868,1263537,100754,156034,253011,243872,256788,496883
1,05,Antioquia,05001,Medellín,2018,Centros Poblados y Rural Disperso,44728,22475,22253,2039,2112,5361,5109,4151,10470
2,05,Antioquia,05001,Medellín,2018,Total,2427133,1141343,1285790,102793,158146,258372,248981,260939,507353
3,05,Antioquia,05002,Abejorral,2018,Cabecera Municipal,7669,3646,4023,526,694,875,820,1220,1695
4,05,Antioquia,05002,Abejorral,2018,Centros Poblados y Rural Disperso,12699,6848,5851,761,561,1843,1765,1322,3608


# Descripción de los datos y transformaciones - Relacion_ips_capacidad_instalada (Tabla 2)

In [8]:

# cargar datos de proyecciones de 2.relacion_ips_capacidad_instalada con utf-8
df_relacion_ips_capacidad_instalada = pd.read_csv(
    config["sources"]["relacion_ips_capacidad_instalada"]["path"],
    sep=';',  # Cambia a punto y coma si usa eso
    encoding='utf-8',
    on_bad_lines='skip'  # Salta líneas problemáticas
)
columnas_texto = ['Departamento','naturaleza','nom descripcion capacidad ']
# Convertir columnas de texto en mayúsculasc con un for 
for col in columnas_texto:
    df_relacion_ips_capacidad_instalada[col] = df_relacion_ips_capacidad_instalada[col].apply(lambda x: x.upper() if isinstance(x, str) else x)

In [9]:
df_relacion_ips_capacidad_instalada.head()

,Departamento,Municipio,Código prestador,Nombre prestador,nit IPS,num digito_verificion,naturaleza,num nivel atencion,Código sede,Número sede,nom sede IPS,Gerente,Dirección,Email,Teléfono,nom grupo capacidad,nom descripcion capacidad,num cantidad capacidad instalada,Fecha Corte,Fuente
0,ANTIOQUIA,APARTADÓ,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,URAMEDICOS,CLAUDIA CECILIA TRUJILLO GOMEZ,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
1,ANTIOQUIA,SALGAR,564204576,ESE HOSPITAL SAN JOSE,"890,981,424",7.0,PÚBLICA,1.0,564204576,1,E.S.E. HOSPITAL SAN JOSE,CESAR AUGUSTO PARDO ROSAS,CL 32A # 33-04,secretaria@hsjsalgar.gov.co,8442020,CAMAS,TPR,2.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
2,AMAZONAS,EL ENCANTO,9100100019,E.S.E. HOSPITAL SAN RAFAEL DE LETICIA,"838,000,096",7.0,PÚBLICA,2.0,9126300019,11,CENTRO DE SALUD SAN RAFAEL - E.S.E HOSPITAL SA...,SAIDA VIVIANA HERREÑO PRIETO,CORREGIMIENTO DEL ENCANTO,sanrafael@esehospitalsanrafael-leticia-amazona...,3203016139,CAMAS,PEDIÁTRICA,3.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
3,AMAZONAS,EL ENCANTO,9100100019,E.S.E. HOSPITAL SAN RAFAEL DE LETICIA,"838,000,096",7.0,PÚBLICA,2.0,9126300019,11,CENTRO DE SALUD SAN RAFAEL - E.S.E HOSPITAL SA...,SAIDA VIVIANA HERREÑO PRIETO,CORREGIMIENTO DEL ENCANTO,sanrafael@esehospitalsanrafael-leticia-amazona...,3203016139,CAMAS,ADULTOS,4.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
4,AMAZONAS,EL ENCANTO,9100100019,E.S.E. HOSPITAL SAN RAFAEL DE LETICIA,"838,000,096",7.0,PÚBLICA,2.0,9126300019,11,CENTRO DE SALUD SAN RAFAEL - E.S.E HOSPITAL SA...,SAIDA VIVIANA HERREÑO PRIETO,CORREGIMIENTO DEL ENCANTO,sanrafael@esehospitalsanrafael-leticia-amazona...,3203016139,CAMAS,TPR,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...


# Descripción de los datos y transformaciones - Divipola (Tabla 4)

In [10]:
# lectura de tabla divipola, forzar a leer todas las columnas en texto
df_divipola = pd.read_excel(config["sources"]["divipola"]["path"], sheet_name=config["sources"]["divipola"]["sheetname"], dtype=str)

### TRANSFORMACIÓN DE LA TABLA 2 RELACION_IPS_CAPACIDAD_INSTALADA con la TABLA 4 DIVIPOLA

 Debido a que la tabla 2. relacion_ips_capacidad_instalada no tiene codigo de departamento ni de municipio pero si tiene los nombres tanto de departamento como municipio, se pueden traer mediante merge para asi poder tener los campos llave correspondientes cruzandolo con la tabla 4.DIVIPOLA

In [11]:
df_capacidad_instalada=df_relacion_ips_capacidad_instalada.copy()

# quitar tildes a las columnas seleccionadas
df_capacidad_instalada[['Departamento','Municipio']] = df_capacidad_instalada[['Departamento','Municipio']].apply(lambda x: x.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8'))


In [12]:
df_dim_divipola=df_divipola.copy()

# quitar tildes a las columnas seleccionadas
df_dim_divipola[['NOMBRE_DPTO','NOMBRE_MUN']] = df_dim_divipola[['NOMBRE_DPTO','NOMBRE_MUN']].apply(lambda x: x.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8'))


In [13]:
# Comparar NOMBRE_DPTO con Departamento
departamentos_divipola = set(df_dim_divipola['NOMBRE_DPTO'].unique())
departamentos_ips = set(df_capacidad_instalada['Departamento'].unique())

print("Departamentos en DIVIPOLA pero NO en IPS:")
print(departamentos_divipola - departamentos_ips)
print("\nDepartamentos en IPS pero NO en DIVIPOLA:")
print(departamentos_ips - departamentos_divipola)
print("\nDepartamentos que SÍ coinciden:")
print(departamentos_divipola & departamentos_ips)


Departamentos en DIVIPOLA pero NO en IPS:
{nan, 'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA', 'BOGOTA, D.C.'}

Departamentos en IPS pero NO en DIVIPOLA:
{'BOGOTA D.C', 'CALI', 'CARTAGENA', 'SAN ANDRES Y PROVIDENCIA', 'BARRANQUILLA', 'SANTA MARTA', 'BUENAVENTURA'}

Departamentos que SÍ coinciden:
{'VICHADA', 'CORDOBA', 'ANTIOQUIA', 'RISARALDA', 'ARAUCA', 'AMAZONAS', 'CESAR', 'HUILA', 'TOLIMA', 'GUAVIARE', 'CALDAS', 'QUINDIO', 'CASANARE', 'PUTUMAYO', 'LA GUAJIRA', 'BOYACA', 'SUCRE', 'META', 'VAUPES', 'MAGDALENA', 'CHOCO', 'CAUCA', 'NARINO', 'ATLANTICO', 'CUNDINAMARCA', 'NORTE DE SANTANDER', 'SANTANDER', 'BOLIVAR', 'VALLE DEL CAUCA', 'GUAINIA', 'CAQUETA'}


In [14]:
## correciones de los departamentos que estan en IPS y no en divipola
valores_cambiar = {'BUENAVENTURA':'VALLE DEL CAUCA',
                   'CALI':'VALLE DEL CAUCA',
                   'BARRANQUILLA':'ATLANTICO',
                   'BOGOTA D.C':'BOGOTA, D.C.',
                   'SANTA MARTA':'MAGDALENA',
                   'CARTAGENA':'BOLIVAR',
                   'SAN ANDRES Y PROVIDENCIA':'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA'
                  }

# Aplicar las correcciones a la columna de departamentos en la tabla df_dim_sede_ips
df_capacidad_instalada['Departamento'] = df_capacidad_instalada['Departamento'].replace(valores_cambiar)

In [15]:
# validacion de los reemplazos
departamentos_divipola = set(df_dim_divipola['NOMBRE_DPTO'].unique())
departamentos_ips = set(df_capacidad_instalada['Departamento'].unique())

print("Departamentos en DIVIPOLA pero NO en IPS:")
print(departamentos_divipola - departamentos_ips)
print("\nDepartamentos en IPS pero NO en DIVIPOLA:")
print(departamentos_ips - departamentos_divipola)
print("\nDepartamentos que SÍ coinciden:")
print(departamentos_divipola & departamentos_ips)


Departamentos en DIVIPOLA pero NO en IPS:
{nan}

Departamentos en IPS pero NO en DIVIPOLA:
set()

Departamentos que SÍ coinciden:
{'VICHADA', 'CORDOBA', 'ANTIOQUIA', 'RISARALDA', 'ARAUCA', 'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA', 'AMAZONAS', 'CESAR', 'HUILA', 'TOLIMA', 'GUAVIARE', 'CALDAS', 'QUINDIO', 'CASANARE', 'PUTUMAYO', 'LA GUAJIRA', 'BOYACA', 'SUCRE', 'META', 'VAUPES', 'MAGDALENA', 'CHOCO', 'CAUCA', 'NARINO', 'ATLANTICO', 'CUNDINAMARCA', 'NORTE DE SANTANDER', 'SANTANDER', 'BOLIVAR', 'VALLE DEL CAUCA', 'GUAINIA', 'BOGOTA, D.C.', 'CAQUETA'}


In [16]:
# Comparar NOMBRE_MUN con Municipio 

municipios_divipola = set(df_dim_divipola['NOMBRE_MUN'].unique())
municipios_ips = set(df_capacidad_instalada['Municipio'].unique())

print("Municipios en DIVIPOLA pero NO en IPS:")
print(municipios_divipola - municipios_ips)
print("\nMunicipios en IPS pero NO en DIVIPOLA:")
print(municipios_ips - municipios_divipola)
print(f"\nTotal municipios que coinciden: {len(municipios_divipola & municipios_ips)}")


Municipios en DIVIPOLA pero NO en IPS:
{'PIENDAMO - TUNIA', 'DONMATIAS', 'SANTIAGO DE CALI', 'GUICAN DE LA SIERRA', 'SAN JUAN DE RIOSECO', 'SOTARA PAISPAMBA', 'NUEVO BELEN DE BAJIRA', 'SANTA CRUZ DE MOMPOX', 'SANTA FE DE ANTIOQUIA', 'SAN JOSE DE CUCUTA', 'PANA PANA', 'MANAURE BALCON DEL CESAR', 'SAN ANDRES DE CUERQUIA', 'SAN VICENTE FERRER', 'MORICHAL', 'SAN ANDRES DE TUMACO', 'BOGOTA, D.C.', 'BARRANCOMINAS', 'LOPEZ DE MICAY', 'PAPUNAHUA', 'CARTAGENA DE INDIAS', nan, 'SAN JOSE DE TOLUVIEJO', 'CERRO DE SAN ANTONIO', 'SAN SEBASTIAN DE MARIQUITA', 'PACOA', 'SAN LUIS DE SINCE', 'PUERTO LEGUIZAMO', 'SAN ANDRES DE SOTAVENTO', 'YAVARATE', 'CACAHUAL', 'SOTARA ', 'PURISIMA DE LA CONCEPCION', 'CUASPUD CARLOSAMA', 'CHIVOLO', 'SAN PEDRO DE LOS MILAGROS', 'LA GUADALUPE'}

Municipios en IPS pero NO en DIVIPOLA:
{'PURISIMA', 'CARTAGENA', 'SOTARA', 'CUASPUD', 'TUMACO', 'GUICAN', 'SAN JUAN DE RIO SECO', 'MOMPOS', 'PIENDAMO', 'TOLU VIEJO', 'SAN ANDRES SOTAVENTO', 'CERRO SAN ANTONIO', 'SINCE', 'LOPEZ', '

In [17]:
# para Municipios en IPS pero NO en DIVIPOLA traer los registros correspondientes
municipios_no_en_divipola = municipios_ips - municipios_divipola
df_municipios_no_en_divipola = df_capacidad_instalada[df_capacidad_instalada['Municipio'].isin(municipios_no_en_divipola)]
df_municipios_no_en_divipola.head()

,Departamento,Municipio,Código prestador,Nombre prestador,nit IPS,num digito_verificion,naturaleza,num nivel atencion,Código sede,Número sede,nom sede IPS,Gerente,Dirección,Email,Teléfono,nom grupo capacidad,nom descripcion capacidad,num cantidad capacidad instalada,Fecha Corte,Fuente
488,"BOGOTA, D.C.",BOGOTA,1100137431,AMBUSURE IPS S.A.S.,"901,457,751",3.0,PRIVADA,NaN,1100137431,1,AMBUSURE IPS,OSCAR ALONSO ROJAS VARGAS,TR 77 C 47-16 P1,ambusureips@gmail.com,3224882104,AMBULANCIAS,BÁSICA,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
851,"BOGOTA, D.C.",BOGOTA,1100137431,AMBUSURE IPS S.A.S.,"901,457,751",3.0,PRIVADA,NaN,1100137431,1,AMBUSURE IPS,OSCAR ALONSO ROJAS VARGAS,TR 77 C 47-16 P1,ambusureips@gmail.com,3224882104,AMBULANCIAS,BÁSICA,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
950,ANTIOQUIA,DON MATIAS,523704806,COOPERATIVA DE TRABAJO ASOCIADO DE PROFESIONAL...,"800,228,773",7.0,PRIVADA,NaN,523704806,4,PROSALCO DON MATIAS,HERNAN DARIO VALLEJO RESTREPO,CL 29 # 28-55,asisgerencia@prosalcoips.com.co,4038424 ext 1123,CONSULTORIOS,CONSULTA EXTERNA,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
951,ANTIOQUIA,DON MATIAS,523704806,COOPERATIVA DE TRABAJO ASOCIADO DE PROFESIONAL...,"800,228,773",7.0,PRIVADA,NaN,523704806,4,PROSALCO DON MATIAS,HERNAN DARIO VALLEJO RESTREPO,CL 29 # 28-55,asisgerencia@prosalcoips.com.co,4038424 ext 1123,SALAS,PROCEDIMIENTOS,3.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...
952,ANTIOQUIA,DON MATIAS,523705948,ESE HOSPITAL FRANCISCO ELADIO BARRERA,"890,905,097",8.0,PÚBLICA,1.0,523705948,1,E.S.E. HOSPITAL FRANCISCO ELADIO BARRERA,OSCAR HERNÁN OCAMPO VARELA,CL 36A # 29-55,subgerencia@hospitaldonmatias.gov.co,8663737,AMBULANCIAS,BÁSICA,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...


In [18]:
df_dim_divipola.head()

,COD_DPTO,NOMBRE_DPTO,COD_MUN,NOMBRE_MUN,COD_CENTRO_POBLADO,NOMBRE_CENTRO_POBLADO,TIPO_CENTRO_POBLADO,LONG,LAT
0,05,ANTIOQUIA,05001,MEDELLIN,05001000,"MEDELLÍN, DISTRITO ESPECIAL DE CIENCIA, TECNOL...",CM,-75.581775,6.246631
1,05,ANTIOQUIA,05001,MEDELLIN,05001001,PALMITAS,CP,-75.690573,6.343919
2,05,ANTIOQUIA,05001,MEDELLIN,05001004,SANTA ELENA,CP,-75.501293,6.210599
3,05,ANTIOQUIA,05001,MEDELLIN,05001009,ALTAVISTA,CP,-75.643706,6.221429
4,05,ANTIOQUIA,05001,MEDELLIN,05001010,AGUAS FRÍAS,CP,-75.633948,6.233335


In [19]:
## correciones de los municipios en la tabla df_dim_sede_ips
valores_cambiar = {'BARRANCO MINAS': 'BARRANCOMINAS',
                   'LEGUIZAMO': 'PUERTO LEGUIZAMO',
                   'LOPEZ': 'LOPEZ DE MICAY',
                   'BOGOTA': 'BOGOTA, D.C.',
                   'PURISIMA': 'PURISIMA DE LA CONCEPCION',
                   'MOMPOS': 'SANTA CRUZ DE MOMPOX',
                   'CALI': 'SANTIAGO DE CALI',
                   'SINCE': 'SAN LUIS DE SINCE',
                   'SAN VICENTE': 'SAN VICENTE FERRER',
                   'SAN ANDRES SOTAVENTO': 'SAN ANDRES DE SOTAVENTO',
                   'TOLU VIEJO': 'SAN JOSE DE TOLUVIEJO',
                   'CUASPUD': 'CUASPUD CARLOSAMA',
                   'CARTAGENA': 'CARTAGENA DE INDIAS',
                   'GUICAN': 'GUICAN DE LA SIERRA',
                   'SANTAFE DE ANTIOQUIA': 'SANTA FE DE ANTIOQUIA',
                   'CERRO SAN ANTONIO': 'CERRO DE SAN ANTONIO',
                   'DON MATIAS': 'DONMATIAS',
                   'CHIBOLO': 'CHIVOLO',
                   'SAN JUAN DE RIO SECO': 'SAN JUAN DE RIOSECO',
                   'PIENDAMO': 'PIENDAMO - TUNIA',
                   'MARIQUITA': 'SAN SEBASTIAN DE MARIQUITA',
                   'CUCUTA': 'SAN JOSE DE CUCUTA',
                   'SOTARA': 'SOTARA ',
                   'TUMACO': 'SAN ANDRES DE TUMACO'
                  }

# Aplicar las correcciones a la columna de municipios en la tabla df_dim_sede_ips
df_capacidad_instalada['Municipio'] = df_capacidad_instalada['Municipio'].replace(valores_cambiar)

In [20]:
# revision NOMBRE_MUN con Municipio 

municipios_divipola = set(df_dim_divipola['NOMBRE_MUN'].unique())
municipios_ips = set(df_capacidad_instalada['Municipio'].unique())

print("Municipios en DIVIPOLA pero NO en IPS:")
print(municipios_divipola - municipios_ips)
print("\nMunicipios en IPS pero NO en DIVIPOLA:")
print(municipios_ips - municipios_divipola)
print(f"\nTotal municipios que coinciden: {len(municipios_divipola & municipios_ips)}")

Municipios en DIVIPOLA pero NO en IPS:
{'PANA PANA', 'MANAURE BALCON DEL CESAR', 'SAN ANDRES DE CUERQUIA', 'PACOA', 'MORICHAL', 'SAN PEDRO DE LOS MILAGROS', 'SOTARA PAISPAMBA', 'NUEVO BELEN DE BAJIRA', 'PAPUNAHUA', 'LA GUADALUPE', 'YAVARATE', 'CACAHUAL', nan}

Municipios en IPS pero NO en DIVIPOLA:
set()

Total municipios que coinciden: 1025


In [21]:
# merge entre df_dim_sede_ips y df_dim_divipola para traer COD_MUN, COD_DPTO tomando como llave Departamento y Municipio
df_capacidad_instalada = df_capacidad_instalada.merge(df_dim_divipola[['COD_MUN', 'COD_DPTO', 'NOMBRE_MUN', 'NOMBRE_DPTO']],
                                        left_on=['Departamento', 'Municipio'],
                                        right_on=['NOMBRE_DPTO', 'NOMBRE_MUN'],
                                        how='left')
df_capacidad_instalada.drop(columns=['NOMBRE_MUN', 'NOMBRE_DPTO'], inplace=True)

In [28]:
df_capacidad_instalada['nom descripcion capacidad '].unique()

<StringArray>
[                             'PROCEDIMIENTOS',
                                         'TPR',
                                  'PEDIÁTRICA',
                                     'ADULTOS',
                          'ATENCIÓN DEL PARTO',
                      'OBSERVACIÓN PEDIÁTRICA',
                 'OBSERVACIÓN ADULTOS HOMBRES',
                 'OBSERVACIÓN ADULTOS MUJERES',
                                   'URGENCIAS',
                            'CONSULTA EXTERNA',
                                      'PARTOS',
                                'UNIDAD MÓVIL',
               'CUIDADO INTERMEDIO PEDIÁTRICO',
                   'CUIDADO INTERMEDIO ADULTO',
                  'CUNA INTERMEDIA PEDIÁTRICA',
                                   'QUIRÓFANO',
                             'SALA DE CIRUGÍA',
                                      'BÁSICA',
                          'INTERMEDIA ADULTOS',
                          'SPA BÁSICO ADULTOS',
                      'SPA

In [22]:
df_capacidad_instalada.head()

,Departamento,Municipio,Código prestador,Nombre prestador,nit IPS,num digito_verificion,naturaleza,num nivel atencion,Código sede,Número sede,...,Dirección,Email,Teléfono,nom grupo capacidad,nom descripcion capacidad,num cantidad capacidad instalada,Fecha Corte,Fuente,COD_MUN,COD_DPTO
0,ANTIOQUIA,APARTADO,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,...,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...,05045,05
1,ANTIOQUIA,APARTADO,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,...,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...,05045,05
2,ANTIOQUIA,APARTADO,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,...,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...,05045,05
3,ANTIOQUIA,APARTADO,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,...,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...,05045,05
4,ANTIOQUIA,APARTADO,504512253,URAMEDICOS S.A.S.,"900,497,151",1.0,PRIVADA,NaN,504512253,1,...,KR 98 # 103-29/37,uramedicos@gmail.com,8286090,SALAS,PROCEDIMIENTOS,1.0,Fecha corte REPS: Nov 5 2022 1:37PM,Base de datos Registro Especial de Prestadores...,05045,05


#  Descripción de los datos y transformaciones - Analisis exploratorio registro prestadores de salud (Tabla 3)

In [23]:
df_registro_prestadores_salud= pd.read_excel(config["sources"]["registro_prestadores_salud"]["path"], sheet_name=config["sources"]["registro_prestadores_salud"]["sheetname"], dtype=str)
df_registro_prestadores_salud.head()

,CodigoPrestador,NombrePrestador,CodigoHabilitacionSede,NombreSede,TipoIdentificacion,NumeroIdentificacion,NaturalezaJuridica,ESE,MunicipioPrestador,DepartamentoPrestadorDesc,...,EmailPrestador,TelefonoPrestador,MunicipioSede,DepartamentoSedeDesc,MunicipioSedeDesc,DireccionSede,EmailSede,TelefonoSede,ClasePrestadorDesc,FechaCorte
0,9100100017,MANUEL DE JESUS LUBO OROZCO,910010001701,MANUEL DE JESUS LUBO OROZCO,CC,72019333,Privada,NO,91001,Amazonas,...,lubomanu@gmail.com,314-3580254,91001,Amazonas,LETICIA,Carrera 6 No. 10-12 Local 2,lubomanu@gmail.com,314-3580254,Profesional Independiente,Fecha corte REPS: Mar 12 2026 3:11PM
1,9100100019,E.S.E. HOSPITAL SAN RAFAEL DE LETICIA,910010001901,E.S.E. HOSPITAL SAN RAFAEL,NI,838000096,P√∫blica,SI,91001,Amazonas,...,gerencia@esehospitalsanrafael-leticia-amazonas...,3203048903-3204650596,91001,Amazonas,LETICIA,CRA 10 # 13-78,calidad@esehospitalsanrafael-leticia-amazonas....,NaN,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Mar 12 2026 3:11PM
2,9100100019,E.S.E. HOSPITAL SAN RAFAEL DE LETICIA,912630001911,CENTRO DE SALUD SAN RAFAEL - E.S.E HOSPITAL SA...,NI,838000096,P√∫blica,SI,91001,Amazonas,...,gerencia@esehospitalsanrafael-leticia-amazonas...,3203048903-3204650596,91263,Amazonas,EL ENCANTO,CORREGIMIENTO DEL ENCANTO,sanrafael@esehospitalsanrafael-leticia-amazona...,3203016139,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Mar 12 2026 3:11PM
3,9100100019,E.S.E. HOSPITAL SAN RAFAEL DE LETICIA,914050001914,CENTRO DE SALUD DE LA CHORRERA- E.S.E HOSPITAL...,NI,838000096,P√∫blica,SI,91001,Amazonas,...,gerencia@esehospitalsanrafael-leticia-amazonas...,3203048903-3204650596,91405,Amazonas,LA CHORRERA,CORREGIMIENTO DE LA CHORRERA,chorrera@esehospitalsanrafael-leticia-amazonas...,3203016132,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Mar 12 2026 3:11PM
4,9100100019,E.S.E. HOSPITAL SAN RAFAEL DE LETICIA,914070001915,CENTRO DE SALUD DE LA PEDRERA - E.S.E HOSPITAL...,NI,838000096,P√∫blica,SI,91001,Amazonas,...,gerencia@esehospitalsanrafael-leticia-amazonas...,3203048903-3204650596,91407,Amazonas,LA PEDRERA,CORREGIMIENTO DE LA PEDRERA,pedrera@esehospitalsanrafael-leticia-amazonas....,3203014882,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Mar 12 2026 3:11PM


In [27]:
df_registro_prestadores_salud.columns

Index(['CodigoPrestador', 'NombrePrestador', 'CodigoHabilitacionSede',
       'NombreSede', 'TipoIdentificacion', 'NumeroIdentificacion',
       'NaturalezaJuridica', 'ESE', 'MunicipioPrestador',
       'DepartamentoPrestadorDesc', 'MunicipioPrestadorDesc',
       'DireccionPrestador', 'EmailPrestador', 'TelefonoPrestador',
       'MunicipioSede', 'DepartamentoSedeDesc', 'MunicipioSedeDesc',
       'DireccionSede', 'EmailSede', 'TelefonoSede', 'ClasePrestadorDesc',
       'FechaCorte'],
      dtype='str')

# Descripción de los datos y transformaciones- df_tiempo (Tabla 5)

In [24]:
df_tiempo= pd.read_csv(config["sources"]["tiempo"]["path"], sep=";", encoding="utf-8", dtype=str)

In [25]:
df_tiempo.head()

#quitar columna dias_con_venta
df_tiempo = df_tiempo.drop(columns=['dias_con_venta'])

#dejar la columna fecha como datetime
df_tiempo['fecha'] = pd.to_datetime(df_tiempo['fecha'])

/var/folders/ym/k0p7fb7s30j873yzly7hngv00000gn/T/ipykernel_18540/612570728.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_tiempo['fecha'] = pd.to_datetime(df_tiempo['fecha'])
